In [3]:
%pip install ollama numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import ollama
import numpy as np

language_model = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"
embedding_model = "hf.co/CompendiumLabs/bge-base-en-v1.5-gguf"

In [5]:
datasets = []

with open("cat-facts.txt", "r", encoding="utf-8") as f:
    datasets = f.readlines()
    print(f"Loaded {len(datasets)} datasets from cat-facts.txt")

Loaded 150 datasets from cat-facts.txt


In [6]:
VECTOR_DB = []

In [7]:
def add_chunk_to_vector_db(chunk):
    embedding = ollama.embed(model = embedding_model, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embedding))

In [8]:
#Vectorization and Storing in VectorDB
for i, chunk in enumerate(datasets):
    add_chunk_to_vector_db(chunk)
    print(f"Added chunk {i+1}/{len(datasets)} to VECTOR_DB")

Added chunk 1/150 to VECTOR_DB
Added chunk 2/150 to VECTOR_DB
Added chunk 3/150 to VECTOR_DB
Added chunk 4/150 to VECTOR_DB
Added chunk 5/150 to VECTOR_DB
Added chunk 6/150 to VECTOR_DB
Added chunk 7/150 to VECTOR_DB
Added chunk 8/150 to VECTOR_DB
Added chunk 9/150 to VECTOR_DB
Added chunk 10/150 to VECTOR_DB
Added chunk 11/150 to VECTOR_DB
Added chunk 12/150 to VECTOR_DB
Added chunk 13/150 to VECTOR_DB
Added chunk 14/150 to VECTOR_DB
Added chunk 15/150 to VECTOR_DB
Added chunk 16/150 to VECTOR_DB
Added chunk 17/150 to VECTOR_DB
Added chunk 18/150 to VECTOR_DB
Added chunk 19/150 to VECTOR_DB
Added chunk 20/150 to VECTOR_DB
Added chunk 21/150 to VECTOR_DB
Added chunk 22/150 to VECTOR_DB
Added chunk 23/150 to VECTOR_DB
Added chunk 24/150 to VECTOR_DB
Added chunk 25/150 to VECTOR_DB
Added chunk 26/150 to VECTOR_DB
Added chunk 27/150 to VECTOR_DB
Added chunk 28/150 to VECTOR_DB
Added chunk 29/150 to VECTOR_DB
Added chunk 30/150 to VECTOR_DB
Added chunk 31/150 to VECTOR_DB
Added chunk 32/15

In [9]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [10]:
def retriever(query, top_k=3):
    query_embedding = ollama.embed(model=embedding_model, input=query)['embeddings'][0]
    similarities = []
    for chunk, embedding in VECTOR_DB:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [11]:
query = "What is the average lifespan of a cat?"

In [12]:
result = retriever(query)
print("Top relevant chunks:")
for i, chunk in enumerate(result):
    print(f"{i+1}. {chunk[0]} (Similarity: {chunk[1]})")

Top relevant chunks:
1. When well treated, a cat can live twenty or more years but the average life span of a domestic cat is 14 years.
 (Similarity: 0.8950300121629418)
2. The oldest cat on record was Crème Puff from Austin, Texas, who lived from 1967 to August 6, 2005, three days after her 38th birthday. A cat typically can live up to 20 years, which is equivalent to about 96 human years.
 (Similarity: 0.775112488931726)
3. On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
 (Similarity: 0.7661794343080901)


In [14]:
context = "\n".join([f"- {chunk}" for chunk, sim in result])

instruction_prompt = f"""You are a helpful assistant that answers 
questions based on the provided context. Use the retrieved chunks 
to formulate your answer. If the information is not sufficient, say you don't know
"""

stream = ollama.chat(
    model=language_model,
    messages=[
        {"role": "system", "content": instruction_prompt},
        {"role": "user", "content": query},
    ],
    stream=True,
)

print("Answer:")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)

Answer:
The average lifespan of a domestic cat is around 12-15 years, but with proper care and attention to health issues, some cats have been known to live up to 20 years. Factors such as breed, lifestyle, diet, and genetics can all impact a cat's lifespan.